In [41]:
import numpy as np
import pandas as pd

In [42]:
from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder
from sklearn.feature_selection import SelectKBest,f_regression,f_classif

In [44]:
df = pd.read_csv("/01 covid_toy - 01 covid_toy.csv")
display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [45]:
print(df.dtypes)

age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object


In [46]:
numeric_columns = df.select_dtypes(include=np.number).columns
print("Numeric Columns:")
print(numeric_columns)

Numeric Columns:
Index(['age', 'fever'], dtype='object')


In [47]:
from pandas.core.arrays import categorical
categorical_columns = df.select_dtypes(include=['object']).columns
print("Categorical Columns:")
print(categorical_columns)

Categorical Columns:
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


In [48]:
df

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No
...,...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore,No
96,51,Female,101.0,Strong,Kolkata,Yes
97,20,Female,101.0,Mild,Bangalore,No
98,5,Female,98.0,Strong,Mumbai,No


In [49]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [50]:
df["fever"] = pd.to_numeric(df["fever"], errors='coerce')
df["fever"] = df["fever"].fillna(df["fever"].mean())
print(df.isnull().sum())

age          0
gender       0
fever        0
cough        0
city         0
has_covid    0
dtype: int64


In [51]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# Dynamically select numeric columns based on df's current state
current_numeric_cols = df.select_dtypes(include=np.number).columns

scaler = MinMaxScaler().set_output(transform="pandas")
scaled_data = scaler.fit_transform(df[current_numeric_cols])
scaled_data.head()

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [52]:
categorical_columns =["origin","car name"]

In [53]:
# Dynamically select categorical columns from the current DataFrame
current_categorical_cols = df.select_dtypes(include=['object']).columns

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")
encoded_data = encoder.fit_transform(df[current_categorical_cols])
display(encoded_data.head())

,gender_Female,gender_Male,cough_Mild,cough_Strong,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai,has_covid_No,has_covid_Yes
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [54]:
X = df.drop("has_covid", axis=1)
Y = df["has_covid"]

In [60]:
X

,age,gender,fever,cough,city
0,60,Male,103.0,Mild,Kolkata
1,27,Male,100.0,Mild,Delhi
2,42,Male,101.0,Mild,Delhi
3,31,Female,98.0,Mild,Kolkata
4,65,Female,101.0,Mild,Mumbai
...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore
96,51,Female,101.0,Strong,Kolkata
97,20,Female,101.0,Mild,Bangalore
98,5,Female,98.0,Strong,Mumbai


In [59]:
Y

,has_covid
0,No
1,Yes
2,No
3,No
4,No
...,...
95,No
96,Yes
97,No
98,No


In [55]:
X_numeric = X.select_dtypes(include=np.number)
# Use f_classif for classification targets and adjust k to the maximum available numeric features (2)
selector = SelectKBest(score_func=f_classif, k=2).set_output(transform="pandas")
X_selected = selector.fit_transform(X_numeric, Y)
display(X_selected.head())

,age,fever
0,60,103.0
1,27,100.0
2,42,101.0
3,31,98.0
4,65,101.0


In [56]:
x_final = pd.concat([X_selected,encoded_data],axis=1)
Selector = SelectKBest(score_func=f_classif, k=5).set_output(transform="pandas")
X_selected = Selector.fit_transform(x_final, Y)
display(X_selected.head())

/usr/local/lib/python3.13/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: divide by zero encountered in divide
  f = msb / msw


,fever,cough_Mild,cough_Strong,has_covid_No,has_covid_Yes
0,103.0,1.0,0.0,1.0,0.0
1,100.0,1.0,0.0,0.0,1.0
2,101.0,1.0,0.0,1.0,0.0
3,98.0,1.0,0.0,1.0,0.0
4,101.0,1.0,0.0,1.0,0.0


In [58]:
# Convert Y from string ('Yes'/'No') to numeric (1/0) for numerical score functions
Y_numeric = Y.map({'Yes': 1, 'No': 0})

selector = SelectKBest(
    score_func=f_classif,
    k=2
).set_output(transform="pandas")

X_selected = selector.fit_transform(X_numeric, Y_numeric)
print(X_selected.head())

   age  fever
0   60  103.0
1   27  100.0
2   42  101.0
3   31   98.0
4   65  101.0
